# 🧬 Hetionet Biomedical Knowledge Graph Ingestion

This notebook loads and explores the **Hetionet** knowledge graph for biomedical link prediction.


## Hetionet Overview
- **47,031 nodes** of 11 types (Compounds, Diseases, Genes, etc.)
- **2,250,197 relationships** of 24 types
- **Key relation for this PoC**: `CtD` = Compound treats Disease

🔗 [Hetionet Project](https://het.io) | 📄 [Original Paper](https://doi.org/10.7554/eLife.26726)


## Goals
1. Load Hetionet edge list
2. Explore graph structure and statistics
3. Extract `CtD` (Compound-treats-Disease) subgraph
4. Sample for PoC scalability (300–500 entities)
5. Export for embedding generation


## 0. Environment Setup


In [ ]:
# Install dependencies (run once)
# !pip install pandas networkx matplotlib seaborn


In [ ]:
import os
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter



In [ ]:
# Plotting configuration
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')


In [ ]:
# Data directory
os.makedirs('../data', exist_ok=True)


## 1. Load Hetionet Edge List


In [ ]:
HETIO_EDGES_URL = 'https://raw.githubusercontent.com/hetio/hetionet/master/hetnet/tsv/hetionet-v1.0-edges.sif'


In [ ]:
def load_hetionet_edges(url: str) -> pd.DataFrame:
    """Load Hetionet edge list from GitHub."""
    df = pd.read_csv(url, sep='\t', names=['source', 'metaedge', 'target'])
    return df


In [ ]:
df_edges = load_hetionet_edges(HETIO_EDGES_URL)
print(f'✅ Loaded {len(df_edges):,} edges from Hetionet')
df_edges.head()


## 2. Explore Graph Structure


In [ ]:
relation_counts = df_edges['metaedge'].value_counts()


In [ ]:
plt.figure(figsize=(12, 6))
relation_counts.head(15).plot(kind='bar')
plt.title('Top 15 Relation Types in Hetionet', fontsize=16, fontweight='bold')
plt.xlabel('Relation Type', fontweight='bold')
plt.ylabel('Count', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
print('Top 10 relation types:')
for rel, count in relation_counts.head(10).items():
    print(f'  {rel}: {count:,}')


## 3. Entity Types


In [ ]:
def extract_entity_type(entity_id: str) -> str:
    """Extract entity type from Hetionet ID (e.g., 'Compound::DB00945' -> 'Compound')."""
    return entity_id.split('::')[0] if '::' in entity_id else 'Unknown'


In [ ]:
df_edges['source_type'] = df_edges['source'].apply(extract_entity_type)


In [ ]:
df_edges['target_type'] = df_edges['target'].apply(extract_entity_type)


In [ ]:
all_entities = pd.concat([df_edges['source_type'], df_edges['target_type']])
entity_counts = all_entities.value_counts()


In [ ]:
plt.figure(figsize=(10, 6))
entity_counts.plot(kind='bar')
plt.title('Entity Type Distribution', fontsize=16, fontweight='bold')
plt.xlabel('Entity Type', fontweight='bold')
plt.ylabel('Count', fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
print('Entity types:')
for entity_type, count in entity_counts.items():
    print(f'  {entity_type}: {count:,}')


## 4. Focus on Drug–Disease Treatment (`CtD`)


In [ ]:
ctd_edges = df_edges[df_edges['metaedge'] == 'CtD'].copy()


In [ ]:
print(f'✅ Found {len(ctd_edges):,} CtD (Compound-treats-Disease) edges')


In [ ]:
print(f"Source types in CtD: {ctd_edges['source_type'].unique()}")


In [ ]:
print(f"Target types in CtD: {ctd_edges['target_type'].unique()}")


In [ ]:
ctd_edges[['source', 'target']].head(10)


## 5. Sample for PoC Scalability


In [ ]:
def sample_kg_for_poc(ctd_edges: pd.DataFrame, max_entities: int = 500, random_state: int = 42) -> pd.DataFrame:
    """Sample a smaller KG for PoC development."""
    all_entities = pd.concat([ctd_edges['source'], ctd_edges['target']]).unique()
    print(f'Total unique entities in CtD: {len(all_entities):,}')
    if len(all_entities) <= max_entities:
        sampled_entities = all_entities
        print(f'Using all {len(all_entities)} entities (≤ {max_entities})')
    else:
        sampled_entities = pd.Series(all_entities).sample(n=max_entities, random_state=random_state).values
        print(f'Sampling {max_entities} entities for PoC scalability')
    sampled_edges = ctd_edges[ctd_edges['source'].isin(sampled_entities) & ctd_edges['target'].isin(sampled_entities)].copy()
    print(f"Sampled KG: {len(sampled_edges):,} edges, {sampled_edges['source'].nunique() + sampled_edges['target'].nunique():,} unique entities")
    return sampled_edges


In [ ]:
sample_300 = sample_kg_for_poc(ctd_edges, max_entities=300)


In [ ]:
sample_500 = sample_kg_for_poc(ctd_edges, max_entities=500)


## 6. Visualize Sampled Subgraph


In [ ]:
import matplotlib.patches as mpatches


In [ ]:
def visualize_subgraph(edges: pd.DataFrame, title: str = 'Sampled KG Subgraph', max_nodes: int = 50):
    # Limit to max_nodes for readability
    if len(edges) > max_nodes:
        edges = edges.sample(n=max_nodes, random_state=42)
    G = nx.DiGraph()
    for _, row in edges.iterrows():
        G.add_edge(row['source'], row['target'], relation=row['metaedge'])
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(G, k=3, iterations=50)
    node_colors = []
    for node in G.nodes():
        if node.startswith('Compound::'):
            node_colors.append('#FF6B6B')
        elif node.startswith('Disease::'):
            node_colors.append('#4ECDC4')
        else:
            node_colors.append('#45B7D1')
    nx.draw(G, pos, node_color=node_colors, node_size=300, with_labels=False, edge_color='gray', alpha=0.7)
    legend_elements = [mpatches.Patch(color='#FF6B6B', label='Compound'), mpatches.Patch(color='#4ECDC4', label='Disease')]
    plt.legend(handles=legend_elements, loc='upper right')
    plt.title(title, fontsize=16, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
visualize_subgraph(sample_300, 'CtD Subgraph (300 entities)')


## 7. Save for Downstream Tasks


In [ ]:
ctd_edges.to_csv('../data/hetionet_ctd_edges.csv', index=False)
print('✅ Saved full CtD edges to ../data/hetionet_ctd_edges.csv')


In [ ]:
sample_300.to_csv('../data/hetionet_ctd_sample_300.csv', index=False)
sample_500.to_csv('../data/hetionet_ctd_sample_500.csv', index=False)
print('✅ Saved PoC samples')

In [ ]:
all_entities_500 = pd.concat([sample_500['source'], sample_500['target']]).unique()
pd.Series(all_entities_500).to_csv('../data/hetionet_entities_500.txt', index=False, header=False)
print(f'✅ Saved {len(all_entities_500)} entities for embedding')


## 🎯 Next Steps
1. **Run `kg_loader.py`** to verify programmatic loading
2. **Proceed to `02-classical-baseline.ipynb`** for embedding generation
3. **Test with real drug-disease pairs**:
   - Dexamethasone (DB00688) → COVID-19 (DOID_0060048)
   - Metformin (DB00316) → Type 2 Diabetes (DOID_9352)
